# Agentic Cyber Security Assistant


**Capstone Project Level:** Intermediate  

### Learning Outcomes
By the end of this lab, you will be able to:

- Understand the architecture of an Agentic AI application.
- Build custom LangChain tools.
- Integrate a Pinecone-backed RAG tool.
- Call an external REST API (NVD CVE API).
- Create multiple agents with `create_agent()`.
- Orchestrate agents using LangGraph.
- Share state between agents.
- Extend the workflow with additional tools and conditional routing.

---

## Project Architecture

```text
                  User
                    │
                    ▼
             Planner Agent
                    │
        ┌───────────┴────────────┐
        ▼                        ▼
 Retrieval Agent          Threat Intel Agent
 (Pinecone RAG)            (NVD CVE API)
        └───────────┬────────────┘
                    ▼
            Validation Agent
                    │
                    ▼
               Final Response
```


## 1. Environment Setup

Install required libraries and configure API keys.


In [ ]:
# !pip install -U langchain langgraph langchain-openai langchain-pinecone pinecone requests python-dotenv

## 2. Imports

Import LangChain, LangGraph, requests and typing.



In [ ]:
import os, requests
from typing import TypedDict
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
# from pinecone import Pinecone
# from langchain_pinecone import PineconeVectorStore


## 3. Configuration

Store API keys and project constants.


In [ ]:
OPENAI_API_KEY='YOUR_OPENAI_KEY'
PINECONE_API_KEY='YOUR_PINECONE_KEY'
NVD_API_KEY='YOUR_NVD_API_KEY'

INDEX_NAME='cyber-security'
NAMESPACE='cis_documents'

os.environ['OPENAI_API_KEY']=OPENAI_API_KEY
llm=ChatOpenAI(model='gpt-4.1-mini',temperature=0)


## 4. Pinecone

Connect to the existing `cyber-security` index using the `cis_documents` namespace.



In [ ]:
# Placeholder
# pc=Pinecone(api_key=PINECONE_API_KEY)
# index=pc.Index(INDEX_NAME)
# retriever=...
print("Connect your existing Pinecone retriever here.")


## 5. RAG Tool

Wrap the retriever inside a LangChain tool.



In [ ]:
@tool
def ask_cis(question:str)->str:
    '''Retrieve CIS benchmark guidance from Pinecone.'''
    # docs=retriever.invoke(question)
    return f"[Placeholder RAG] {question}"


## 6. CVE Tool

Call the NVD REST API and parse JSON.



In [ ]:
@tool
def lookup_cve(keyword:str)->str:
    url='https://services.nvd.nist.gov/rest/json/cves/2.0'
    params={'keywordSearch':keyword,'resultsPerPage':3}
    headers={'apiKey':NVD_API_KEY} if NVD_API_KEY else {}
    try:
        r=requests.get(url,params=params,headers=headers,timeout=20)
        r.raise_for_status()
        data=r.json()
        out=[]
        for item in data.get('vulnerabilities',[]):
            c=item['cve']
            out.append(f"{c['id']} : {c['descriptions'][0]['value'][:150]}")
        return "\n".join(out) if out else "No results"
    except Exception as e:
        return str(e)


## 7. Planner Agent

Decides which information is required.



In [ ]:
planner_agent=create_agent(
    model=llm,
    tools=[],
    system_prompt="""You are the Planner Agent.

Return ONLY JSON:
{
  "use_rag": true|false,
  "use_cve": true|false,
  "reason":"..."
}

Rules:
- Configuration, CIS, hardening, benchmark, policy -> use_rag=true
- Latest, CVE, vulnerability, exploit -> use_cve=true
- If both best practices and recent threats are requested -> both true.
"""
)


## 8. Retrieval Agent

Uses the RAG tool.



In [ ]:
retrieval_agent=create_agent(
    model=llm,
    tools=[ask_cis],
    system_prompt="You are the Retrieval Agent."
)


## 9. Threat Agent

Uses the CVE lookup tool.



In [ ]:
threat_agent=create_agent(
    model=llm,
    tools=[lookup_cve],
    system_prompt="You are the Threat Agent."
)


## 10. Validator Agent

Combines evidence into a reliable answer.



In [ ]:
validator_agent=create_agent(
    model=llm,
    tools=[],
    system_prompt="You are the Validator Agent."
)


## 11. State

Define the shared graph state.


In [ ]:
class CyberState(TypedDict):
    query:str
    plan:str
    rag:str
    cves:str
    final:str


## 12. Graph

Wire agents together using LangGraph.



In [ ]:
import json

def planner_node(state):
    # Production: parse planner JSON. Teaching fallback below.
    q=state["query"].lower()
    use_rag=any(k in q for k in ["cis","benchmark","harden","secure","configure","policy","password","firewall"])
    use_cve=any(k in q for k in ["cve","latest","vulnerab","exploit","threat"])
    if "secure" in q and use_cve:
        use_rag=True
    return {"query":state["query"],"use_rag":use_rag,"use_cve":use_cve}

def rag_node(state):
    return {"rag":"" if not state["use_rag"] else str(retrieval_agent.invoke({"messages":[{"role":"user","content":state["query"]}]}))}

def cve_node(state):
    return {"cves":"" if not state["use_cve"] else str(threat_agent.invoke({"messages":[{"role":"user","content":state["query"]}]}))}

def validator_node(state):
    prompt=f'''Query:{state["query"]}

RAG:
{state.get("rag","")}

CVEs:
{state.get("cves","")}
'''
    return {"final":str(validator_agent.invoke({"messages":[{"role":"user","content":prompt}]}))}

def planner_router(state):
    routes=[]
    if state["use_rag"]:
        routes.append("rag")
    if state["use_cve"]:
        routes.append("cve")
    if not routes:
        routes=["rag"]
    return routes

g=StateGraph(CyberState)
g.add_node("planner",planner_node)
g.add_node("rag",rag_node)
g.add_node("cve",cve_node)
g.add_node("validator",validator_node)

g.add_edge(START,"planner")
g.add_conditional_edges("planner",planner_router,{"rag":"rag","cve":"cve"})
# fan-in: both branches converge
g.add_edge("rag","validator")
g.add_edge("cve","validator")
g.add_edge("validator",END)
app=g.compile()


## 13. Execution

Invoke the graph on a user query.



In [ ]:
result=app.invoke({'query':'How do I harden Windows SMB against ransomware?'})
print(result['final'])


## 14. Visualization

Render the LangGraph Mermaid diagram.



In [ ]:
from IPython.display import Image,display
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(e)


## 15. Exercises

Suggested enhancements and assignments.



### Suggested Assignments

- see activities\capstone.md


# Appendix

## Discussion Questions

- Why do we use tools instead of placing all logic inside one agent?
- What benefits does LangGraph provide over sequential execution?
- When should an agent call an external API instead of relying on RAG?
- How can hallucinations be reduced using validation loops?

## Best Practices

- Keep tools single-purpose.
- Keep prompts concise and explicit.
- Share only necessary state.
- Validate external data before presenting it.
- Log agent decisions for debugging and teaching.
